# 01 — Análisis del foro — Exploit.in

Análisis estructurado del foro sin LLM: jerarquía de usuarios, sistema de reputación y marketplace.

| Sección | Qué se analiza |
|---|---|
| 1. Jerarquía de usuarios | Grupos, actividad, registro temporal |
| 2. Sistema de reputación | Red de confianza, positivo vs negativo, comentarios |
| 3. Marketplace | Qué se compra/vende, quién, cuándo |
| 4. Black List | Casos de fraude documentados por la comunidad |
| 5. Secciones de acceso restringido | 1st/2nd Access Level — usuarios privilegiados |

## 0. Setup

In [ ]:
# Importamos las librerías necesarias para el análisis.

# "pandas" para trabajar con tablas de datos (DataFrames).
import pandas as pd

# "numpy" es la librería base para operaciones matemáticas y numéricas en Python.
# La importamos como "np" por convención.
import numpy as np

# "matplotlib.pyplot" para crear gráficas básicas.
import matplotlib.pyplot as plt

# "matplotlib.ticker" para personalizar el formato de los números en los ejes.
import matplotlib.ticker as mticker

# "matplotlib.cm" contiene paletas de colores predefinidas (color maps).
import matplotlib.cm as cm

# "pathlib.Path" para manejar rutas de archivos de forma segura.
from pathlib import Path

# "IPython.display.display" para mostrar tablas formateadas en el notebook.
from IPython.display import display

# Definimos la carpeta donde están los archivos Parquet generados por el notebook 00.
PROCESSED = Path('../data/processed')

# Cargamos cada tabla desde su archivo Parquet.
# "pd.read_parquet()" lee un archivo Parquet y lo convierte en un DataFrame de pandas.
# Es mucho más rápido que leer el CSV original porque los Parquet están optimizados.
posts          = pd.read_parquet(PROCESSED / 'posts.parquet')
topics         = pd.read_parquet(PROCESSED / 'topics.parquet')
forums         = pd.read_parquet(PROCESSED / 'forums.parquet')
members        = pd.read_parquet(PROCESSED / 'members.parquet')
message_text   = pd.read_parquet(PROCESSED / 'message_text.parquet')
message_topics = pd.read_parquet(PROCESSED / 'message_topics.parquet')
reputation     = pd.read_parquet(PROCESSED / 'reputation.parquet')

# Creamos diccionarios de "traducción" para convertir IDs numéricos en nombres legibles.
# Estos se usan muchas veces a lo largo del notebook, por eso los definimos aquí al principio.
forum_name_map  = dict(zip(forums['id'], forums['name']))      # id_foro → nombre_foro
topic_forum_map = dict(zip(topics['tid'], topics['forum_id']))  # id_hilo → id_foro
member_name_map = dict(zip(members['id'], members['name']))     # id_miembro → nombre

# Añadimos a cada post el nombre de la sección del foro a la que pertenece.
# Hacemos una copia para no modificar el DataFrame original (buena práctica en pandas).
posts = posts.copy()
posts['forum_id']   = posts['topic_id'].map(topic_forum_map)  # hilo → foro (id)
posts['forum_name'] = posts['forum_id'].map(forum_name_map)   # foro (id) → foro (nombre)

# Diccionario que traduce los códigos numéricos de grupo de IPB (Invision Power Board)
# a nombres legibles en ruso. En IPB, cada usuario pertenece a un grupo que determina
# sus permisos y su rango visible en el foro.
GROUP_NAMES = {
    1:   'Ожидающий',      # Pendiente de activación
    2:   'Гость',          # Invitado (sin cuenta)
    3:   'Пользователь',   # Usuario regular
    4:   'Админ',          # Administrador
    6:   'Супермодератор',  # Supermoderador
    7:   'Модератор',      # Moderador
    8:   'Забанен',        # Baneado (cuenta suspendida)
    100: 'Доверенный',     # Usuario de confianza especial
}

# Añadimos el nombre del grupo a la tabla de miembros usando el diccionario anterior.
# "fillna('Desconocido')" rellena los grupos que no están en el diccionario.
members = members.copy()
members['group_name'] = members['mgroup'].map(GROUP_NAMES).fillna('Desconocido')

# Verificamos que todo se cargó correctamente mostrando el número de filas de cada tabla.
print('Tablas cargadas:')
for name, df in [('posts', posts), ('topics', topics), ('forums', forums),
                  ('members', members), ('message_text', message_text),
                  ('reputation', reputation)]:
    print(f'  {name:20s} {len(df):8,} filas')

## 1. Jerarquía de usuarios

In [ ]:
# Calculamos estadísticas por grupo de usuarios:
# - "usuarios": cuántos usuarios hay en cada grupo (count de 'id')
# - "posts_totales": suma de todos los posts de usuarios en ese grupo
# - "rep_media": promedio de reputación de los usuarios en ese grupo
# ".agg()" permite aplicar múltiples funciones de agregación a la vez.
group_dist = members.groupby('group_name').agg(
    usuarios=('id', 'count'),
    posts_totales=('posts', 'sum'),
    rep_media=('rep', 'mean'),
).sort_values('usuarios', ascending=False)

print('Distribución por grupo:')
display(group_dist)

# Creamos una figura con dos gráficas lado a lado.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Asignamos colores según el tipo de grupo: rojo para baneados, verde para confianza,
# naranja para roles de moderación, y azul para el resto.
colors = ['#c0392b' if g == 'Забанен' else '#27ae60' if g == 'Доверенный'
          else '#e67e22' if g in ('Админ','Модератор','Супермодератор')
          else '#3498db' for g in group_dist.index]

# Gráfica izquierda: número de usuarios por grupo.
# "[::-1]" invierte el orden para que matplotlib muestre la barra más grande arriba.
axes[0].barh(group_dist.index[::-1], group_dist['usuarios'][::-1], color=colors[::-1], alpha=0.85)
axes[0].set_title('Usuarios por grupo')
axes[0].set_xlabel('Usuarios')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Gráfica derecha: posts totales por grupo.
axes[1].barh(group_dist.index[::-1], group_dist['posts_totales'][::-1], color=colors[::-1], alpha=0.85)
axes[1].set_title('Posts totales por grupo')
axes[1].set_xlabel('Posts')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# "suptitle" añade un título global que abarca las dos gráficas.
plt.suptitle('Exploit.in — Jerarquía de grupos de usuarios', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Filtramos los miembros que tienen fecha de registro conocida.
# ".dropna(subset=['joined'])" elimina filas donde 'joined' sea nulo.
members_dated = members.dropna(subset=['joined']).copy()

# Creamos una columna con el mes y año de registro de cada usuario.
# ".dt.to_period('M')" convierte la fecha exacta en un periodo mensual.
members_dated['month'] = members_dated['joined'].dt.to_period('M')

# Contamos cuántos usuarios se registraron cada mes.
monthly_reg = members_dated.groupby('month').size()

# Convertimos los periodos a timestamps para graficarlos en eje de tiempo.
monthly_reg.index = monthly_reg.index.to_timestamp()

# Creamos la gráfica de evolución temporal de registros.
fig, ax = plt.subplots(figsize=(13, 4))

# Área sombreada bajo la curva (alpha bajo = muy transparente).
ax.fill_between(monthly_reg.index, monthly_reg.values, alpha=0.25, color='#3498db')

# Línea de la curva encima del área.
ax.plot(monthly_reg.index, monthly_reg.values, color='#3498db', lw=1.5)

ax.set_title('Exploit.in — Nuevos registros por mes')
ax.set_ylabel('Usuarios registrados')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

# Identificamos el mes con mayor número de registros nuevos.
peak = monthly_reg.idxmax()
print(f'Pico de registros: {peak.strftime("%Y-%m")} — {monthly_reg.max():,} usuarios')
print(f'Total usuarios con fecha: {len(members_dated):,}')

In [ ]:
# Obtenemos el top 25 de usuarios más activos del foro (por número de posts).
# Solo incluimos usuarios con al menos 1 post para descartar cuentas inactivas.
power_users = (
    members[members['posts'] > 0]
    [['name', 'group_name', 'posts', 'rep', 'joined']]  # columnas de interés
    .sort_values('posts', ascending=False)               # ordenamos de más a menos posts
    .head(25)                                            # tomamos solo los 25 primeros
    .reset_index(drop=True)
)
# Empezamos el índice en 1 para que el ranking sea más natural (1er lugar, 2do lugar, etc.)
power_users.index += 1
print('Top 25 usuarios por actividad:')
display(power_users)

# Obtenemos los usuarios baneados (mgroup == 8) con más posts antes de ser baneados.
# Esto es interesante desde el punto de vista de seguridad: los usuarios problemáticos
# a menudo fueron muy activos antes de que los administradores los banearan.
banned = (
    members[members['mgroup'] == 8][['name', 'posts', 'rep', 'joined']]
    .sort_values('posts', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
banned.index += 1
print(f'\nUsuarios baneados con más posts previos (top 10):')
display(banned)

## 2. Sistema de reputación

En IPB la reputación es la moneda de confianza del foro:  
`CODE='01'` = voto positivo · `CODE='02'` = voto negativo

In [ ]:
# Creamos una copia de la tabla de reputación para no modificar la original.
rep = reputation.copy()

# Añadimos los nombres de quien dio y quien recibió el voto,
# usando el diccionario de traducción id → nombre.
rep['giver']    = rep['from_id'].map(member_name_map)
rep['receiver'] = rep['member_id'].map(member_name_map)

# Creamos una columna booleana: True si es voto positivo (CODE='01'), False si es negativo.
rep['positive'] = rep['CODE'] == '01'

# Calculamos la reputación neta (positivos - negativos) para cada usuario.
# Primero contamos los votos positivos recibidos por cada usuario.
rep_pos = rep[rep['positive']].groupby('receiver').size().rename('pos')

# Luego contamos los votos negativos.
rep_neg = rep[~rep['positive']].groupby('receiver').size().rename('neg')
# "~" es el operador de negación booleana: ~True = False, ~False = True.

# Unimos ambas Series en un DataFrame compartiendo el índice (nombre del receptor).
# ".fillna(0)" rellena con 0 los usuarios que no tienen votos positivos o negativos.
# ".astype(int)" convierte los decimales que genera fillna (0.0) a enteros.
rep_net = pd.concat([rep_pos, rep_neg], axis=1).fillna(0).astype(int)

# Calculamos la reputación neta: positivos menos negativos.
rep_net['net'] = rep_net['pos'] - rep_net['neg']

# Ordenamos de mayor a menor reputación neta para ver quiénes son los más confiables.
rep_net = rep_net.sort_values('net', ascending=False)

print('=== REPUTACIÓN NETA (top/bottom 10) ===')
print('\nMás confiables:')
display(rep_net.head(10))

print('\nMenos confiables:')
display(rep_net.tail(10))

In [ ]:
# Creamos una figura con dos gráficas lado a lado para comparar
# los usuarios con mejor y peor reputación.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Gráfica izquierda: top 15 usuarios con más reputación neta positiva.
top15 = rep_net.head(15)
axes[0].barh(top15.index[::-1], top15['net'][::-1], color='#27ae60', alpha=0.85)
axes[0].set_title('Top 15 — Mayor reputación neta')
axes[0].set_xlabel('Votos positivos − negativos')

# Gráfica derecha: top 15 usuarios con reputación más negativa.
# Solo incluimos usuarios que tienen al menos un voto negativo (neg > 0).
# Ordenamos de menor a mayor (los más negativos primero) y tomamos los 15 peores.
bot15 = rep_net[rep_net['neg'] > 0].sort_values('net').head(15)
axes[1].barh(bot15.index[::-1], bot15['net'][::-1], color='#c0392b', alpha=0.85)
axes[1].set_title('Top 15 — Reputación más negativa')
axes[1].set_xlabel('Votos positivos − negativos')

plt.suptitle('Exploit.in — Red de confianza (sistema de reputación)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Calculamos quién es más activo dando votos de reputación (los "árbitros" del foro).
# Agrupamos por el que dio el voto ("giver") y calculamos:
# - total: cuántos votos dio en total
# - positivos: cuántos de esos votos fueron positivos
# ".assign()" añade una nueva columna usando las existentes. "d" es el DataFrame en construcción.
top_givers = rep.groupby('giver').agg(
    total=('positive', 'count'),
    positivos=('positive', 'sum'),
).assign(negativos=lambda d: d['total'] - d['positivos'])

# Ordenamos de más a menos activo y mostramos los 15 que más votos han dado.
top_givers = top_givers.sort_values('total', ascending=False).head(15)

print('Top 15 usuarios que más votos de reputación han dado:')
display(top_givers)

# Filtramos los votos que tienen comentario de texto con más de 5 caracteres
# (para excluir los comentarios vacíos o con solo espacios).
rep_comentados = rep[rep['message'].notna() & (rep['message'].str.len() > 5)].copy()
print(f'\nVotos con comentario: {len(rep_comentados):,} de {len(rep):,} ({len(rep_comentados)/len(rep)*100:.0f}%)')

# Mostramos una muestra aleatoria de comentarios de reputación para ver
# el tipo de lenguaje y razones que los usuarios del foro usaban para calificarse.
print('\nMuestra de comentarios:')
sample_rep = rep_comentados.sample(8, random_state=42)[['giver','receiver','positive','message']]
for _, row in sample_rep.iterrows():
    # Usamos un triángulo como indicador visual del tipo de voto.
    icon = '▲' if row['positive'] else '▼'
    # Mostramos dador → receptor y los primeros 90 caracteres del comentario.
    print(f'  {icon} {str(row["giver"]):15s} → {str(row["receiver"]):15s}: {str(row["message"])[:90]}')

In [ ]:
# Creamos un "heatmap" (mapa de calor) que muestra cuántos votos positivos se dieron
# entre los 20 usuarios que más votos dieron y los 20 que más recibieron.
# Este tipo de visualización revela la estructura de la red de confianza del foro.

# Obtenemos los 20 usuarios que más votos dieron (los "árbitros").
top_g = rep['giver'].value_counts().head(20).index.tolist()

# Obtenemos los 20 usuarios que más votos recibieron.
top_r = rep['receiver'].value_counts().head(20).index.tolist()

# Filtramos los votos de reputación para quedarnos solo con los intercambios
# entre esos 20 dadores y esos 20 receptores.
rep_sub = rep[rep['giver'].isin(top_g) & rep['receiver'].isin(top_r)]

# Creamos una tabla dinámica (pivot table) que cuenta los votos positivos
# de cada dador (filas) a cada receptor (columnas).
# ".unstack()" convierte el índice interno (receiver) en columnas del DataFrame.
# "fill_value=0" rellena con 0 los pares que nunca se votaron.
heat = rep_sub.groupby(['giver','receiver'])['positive'].sum().unstack(fill_value=0)

# Reindexamos para asegurarnos de que están todos los usuarios, incluso los que
# no interactuaron (con 0 en esas celdas).
heat = heat.reindex(index=top_g, columns=top_r, fill_value=0)

# Creamos el heatmap con imshow(), que muestra la matriz de valores como una imagen
# donde el color indica la intensidad (0 = blanco, alto = rojo oscuro).
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(heat.values, cmap='YlOrRd', aspect='auto')

# Configuramos las etiquetas de los ejes con los nombres de los usuarios.
ax.set_xticks(range(len(top_r)))
ax.set_xticklabels(top_r, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(top_g)))
ax.set_yticklabels(top_g, fontsize=8)

ax.set_xlabel('Receptor →')
ax.set_ylabel('← Dador')
ax.set_title('Red de reputación — votos positivos\n(top 20 dadores × top 20 receptores)')

# "colorbar" añade la barra de colores a la derecha que explica la escala de valores.
# "fraction=0.02" controla el ancho de la barra de colores relativo a la gráfica.
plt.colorbar(im, ax=ax, fraction=0.02)
plt.tight_layout()
plt.show()

## 3. Marketplace — Покупка/Продажа/Обмен/Работа

La sección más activa del foro (2,627 hilos, 7,750 posts).  
Plataforma de compraventa de credenciales, shells, accesos, ICQ, servicios de spam, etc.

In [ ]:
# Nombre de la sección del marketplace en ruso.
market_name = 'Покупка/Продажа/Обмен/Работа'

# Filtramos los posts que pertenecen a esa sección del foro.
market = posts[posts['forum_name'] == market_name].copy()

print(f'Posts en marketplace: {len(market):,}')
# ".nunique()" cuenta los valores únicos (número de hilos distintos en el marketplace).
print(f'Hilos únicos        : {market["topic_id"].nunique():,}')
print(f'Vendedores/Compradores únicos: {market["author_name"].nunique():,}')

# Creamos una columna con el mes de cada post para analizar la evolución temporal.
market_dated = market.dropna(subset=['post_date']).copy()
market_dated['month'] = market_dated['post_date'].dt.to_period('M')
market_monthly = market_dated.groupby('month').size()
market_monthly.index = market_monthly.index.to_timestamp()

# Gráfica de actividad mensual del marketplace con área rellena en púrpura.
fig, ax = plt.subplots(figsize=(13, 4))
ax.fill_between(market_monthly.index, market_monthly.values, alpha=0.3, color='#8e44ad')
ax.plot(market_monthly.index, market_monthly.values, color='#8e44ad', lw=1.5)
ax.set_title('Exploit.in — Actividad mensual del Marketplace')
ax.set_ylabel('Posts')
plt.tight_layout()
plt.show()

In [ ]:
# Calculamos los 20 usuarios más activos en el marketplace.
# Agrupamos por autor y calculamos dos métricas:
# - "posts": número total de posts (respuestas incluidas)
# - "hilos_iniciados": número de hilos que el usuario abrió (new_topic == True)
#   Los hilos propios suelen ser los anuncios de compra/venta del usuario.
top_market = (
    market.groupby('author_name').agg(
        posts=('pid', 'count'),
        hilos_iniciados=('new_topic', 'sum'),
    )
    .sort_values('posts', ascending=False)
    .head(20)
)

# Añadimos la reputación de cada usuario para cruzar actividad con confianza.
# Creamos un diccionario temporal nombre → reputación y lo aplicamos con ".map()".
top_market['rep'] = top_market.index.map(
    dict(zip(members['name'], members['rep']))
)

print('Top 20 usuarios más activos en el marketplace:')
display(top_market)

# Visualizamos los 20 usuarios más activos en el marketplace con barras horizontales.
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_market.index[::-1], top_market['posts'][::-1], color='#8e44ad', alpha=0.85)
ax.set_title('Exploit.in — Top 20 usuarios en Marketplace')
ax.set_xlabel('Posts en sección Покупка/Продажа/Обмен/Работа')
plt.tight_layout()
plt.show()

In [ ]:
# Definimos categorías de productos/servicios que se venden en el marketplace,
# junto con palabras clave en ruso e inglés que indican cada categoría.
# Muchos términos técnicos del underground hacker se usan en inglés incluso en foros rusos.
keywords = {
    'shells / accesos':    ['shell', 'шелл', 'rdp', 'ssh', 'доступ', 'сервер'],
    'ICQ / contactos':     ['icq', 'аська', 'контакт', 'номер'],
    'carding / dumps':     ['карт', 'dump', 'cvv', 'cc ', 'кард'],
    'spam / mailing':      ['спам', 'рассылк', 'база', 'мейл', 'mail'],
    'malware / exploits':  ['trojan', 'эксплойт', 'бот', 'крипт', 'stub'],
    'passwords / accounts':['пароль', 'аккаунт', 'login', 'pass', 'акк'],
}

# Convertimos el contenido de todos los posts del marketplace a minúsculas
# para que la búsqueda no distinga entre mayúsculas y minúsculas.
# ".fillna('')" reemplaza los valores nulos con cadenas vacías para evitar errores.
market_content = market['content'].str.lower().fillna('')

print('Posts por categoría de producto (búsqueda por palabras clave):')
for cat, words in keywords.items():
    # Para cada categoría, creamos una máscara booleana que indica qué posts
    # contienen al menos una de las palabras clave.
    # "|".join(words) une las palabras con "|" (operador OR de expresiones regulares).
    # Por ejemplo: 'shell|шелл|rdp|ssh|доступ|сервер'
    mask = market_content.str.contains('|'.join(words), na=False)
    # ".sum()" cuenta los True de la máscara (los posts que contienen al menos una palabra clave).
    print(f'  {cat:30s}: {mask.sum():,} posts')

# Mostramos una muestra de 4 posts reales del marketplace para tener contexto.
# Solo posts con contenido de más de 50 chars para evitar respuestas muy breves.
print('\n=== MUESTRA DE POSTS DEL MARKETPLACE ===')
sample_market = market[market['content'].str.len() > 50].sample(4, random_state=7)
for _, row in sample_market.iterrows():
    date_str = row['post_date'].strftime('%Y-%m-%d') if pd.notna(row['post_date']) else 'N/A'
    print(f'\n[{row["author_name"]} · {date_str}]')
    # Mostramos los primeros 400 caracteres del post para no saturar la pantalla.
    print(row['content'][:400])

## 4. Black List — Registro de estafadores

La comunidad documenta a sus propios estafadores.  
Sección de 73 hilos, 554 posts — registro de fraudes entre miembros.

In [ ]:
# Filtramos los posts de la sección "Black List" (Lista negra de estafadores)
# y de "White List" (Lista blanca de vendedores confiables).
blacklist = posts[posts['forum_name'] == 'Black List'].copy()
whitelist = posts[posts['forum_name'] == 'White List'].copy()

print(f'Black List: {len(blacklist):,} posts en {blacklist["topic_id"].nunique()} hilos')
print(f'White List: {len(whitelist):,} posts en {whitelist["topic_id"].nunique()} hilos')

# Obtenemos los títulos de los hilos de la sección Black List.
# Primero buscamos el ID de la sección "Black List" en la tabla de foros.
# Luego filtramos los hilos (topics) que pertenecen a ese foro.
bl_topics = topics[topics['forum_id'].isin(
    forums[forums['name'] == 'Black List']['id'].tolist()
)][['tid','title','posts','views','starter_name']].sort_values('posts', ascending=False)

print('\nHilos más activos en Black List:')
display(bl_topics.head(15).reset_index(drop=True))

# Mostramos los primeros mensajes de cada hilo (los que abrieron el hilo, new_topic == 1).
# Estos mensajes suelen contener la denuncia completa contra el estafador,
# incluyendo logs de conversaciones, capturas de pantalla y datos del acusado.
print('\nMuestra de posts (primeros mensajes de cada hilo):')
bl_sample = blacklist[blacklist['new_topic'] == 1].head(5)
for _, row in bl_sample.iterrows():
    print(f'\n  [{row["author_name"]}] {str(row["content"])[:300]}')

## 5. Secciones de acceso restringido

`1st Access Level` y `2nd Access Level` son secciones premium que solo ven usuarios de confianza.  
El dump las incluye completas — 5,719 posts en total.

In [ ]:
# Filtramos los posts de las secciones de acceso restringido.
# Estas secciones solo eran visibles para usuarios con nivel de confianza elevado.
# El dump SQL las incluye completas, lo que permite analizar su contenido.
restricted = posts[posts['forum_name'].isin(['1st Access Level', '2nd Access Level'])].copy()

print(f'Posts en secciones restringidas: {len(restricted):,}')

# Calculamos estadísticas por sección: posts totales, hilos únicos y autores únicos.
print(restricted.groupby('forum_name').agg(
    posts=('pid','count'),
    hilos=('topic_id','nunique'),
    autores=('author_name','nunique')
).to_string())

# Identificamos qué usuarios publicaron más en las secciones premium.
# Esto nos indica quiénes tenían el nivel de acceso más alto en el foro.
top_restricted = (
    restricted.groupby('author_name').size()
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
)
top_restricted.columns = ['usuario', 'posts_premium']

# Añadimos la reputación y el grupo de cada usuario para enriquecer el análisis.
top_restricted['rep'] = top_restricted['usuario'].map(
    dict(zip(members['name'], members['rep']))
)
top_restricted['grupo'] = top_restricted['usuario'].map(
    dict(zip(members['name'], members['group_name']))
)
top_restricted.index += 1

print('\nTop 15 usuarios con más posts en secciones premium:')
display(top_restricted)

# Mostramos una muestra del contenido de las secciones premium.
# Los posts aquí suelen contener información más sensible y de mayor valor técnico.
print('\nMuestra de posts en 1st Access Level:')
sample_1st = (
    restricted[restricted['forum_name'] == '1st Access Level']
    [restricted['content'].str.len() > 80]
    .sample(3, random_state=42)
)
for _, row in sample_1st.iterrows():
    date_str = row['post_date'].strftime('%Y-%m-%d') if pd.notna(row['post_date']) else 'N/A'
    print(f'\n[{row["author_name"]} · {date_str}]')
    print(row['content'][:350])

## 6. Resumen — Perfil de la comunidad

In [ ]:
# Generamos un resumen final con los indicadores más importantes de la comunidad.
print('='*58)
print('EXPLOIT.IN — PERFIL DE LA COMUNIDAD')
print('='*58)

# Obtenemos los nombres de los administradores (mgroup == 4) y usuarios de confianza (mgroup == 100).
# ".tolist()" convierte la Serie de pandas a una lista de Python normal.
admins  = members[members['mgroup'] == 4]['name'].tolist()
trusted = members[members['mgroup'] == 100]['name'].tolist()

# Contamos el número de usuarios baneados (mgroup == 8).
banned_count = (members['mgroup'] == 8).sum()

print(f'Administradores       : {admins}')
print(f'Usuarios de confianza : {trusted}')
print(f'Usuarios baneados     : {banned_count:,} ({banned_count/len(members)*100:.1f}%)')
print()

# Estadísticas del sistema de reputación: cuántos tienen saldo positivo y cuántos negativo.
print(f'Usuarios con reputación positiva neta : {(rep_net["net"] > 0).sum():,}')
print(f'Usuarios con reputación negativa neta : {(rep_net["net"] < 0).sum():,}')
print()

# Resumen de actividad por sección clave.
print(f'Posts en marketplace  : {len(market):,} ({len(market)/len(posts)*100:.1f}% del total)')
print(f'Posts en secciones TI : {len(posts[posts["forum_name"].isin(["Безопасность и взлом","Деньги","Вирусология"])]):,}')
print(f'Posts en Black List   : {len(blacklist):,}')
print(f'Posts premium (1st+2nd): {len(restricted):,}')
print()

# Mostramos el usuario con mayor reputación neta (el más confiable del foro).
print('Usuario con mayor reputación:')
top_rep = rep_net.head(1)
print(f'  {top_rep.index[0]} — net={top_rep["net"].iloc[0]:+d} (pos={top_rep["pos"].iloc[0]}, neg={top_rep["neg"].iloc[0]})')